# 1. Exploratory Data Analysis & Hospital Readmission Risk

### Problem Statement
The objective of this project is to analyze the authentic **UCI Diabetes 130-US Hospitals (1999–2008)** dataset and estimate the risk of an unplanned hospital readmission within 30 days of discharge (`readmitted == '<30'`).

### Dataset Information:
- **Source:** UCI Machine Learning Repository
- **Volume:** 101,766 hospital encounters across 130 clinical facilities
- **Attributes:** 50 raw features covering patient demographics, clinical diagnoses (ICD-9), inpatient procedures, lab tests (HbA1c, glucose), 23 diabetes medications, and prior 12-month healthcare utilization.


### 1. Import Data and Required Packages
Importing Pandas, Numpy, Matplotlib, Seaborn, Scipy, and Warnings Library.


In [ ]:
import os
import sys
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import scipy.stats as stats
import matplotlib.pyplot as plt
import seaborn as sns

# Visual formatting
sns.set_theme(style="whitegrid", font="sans-serif")
plt.rcParams.update({
    'figure.autolayout': True,
    'figure.titlesize': 14,
    'axes.titlesize': 12,
    'axes.labelsize': 11
})


### 2. Import the CSV Data as Pandas DataFrames
Reading raw encounter dataset (`diabetic_data.csv`) and administrative mappings (`IDS_mapping.csv`).


In [ ]:
# Resolve project paths
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
raw_data_path = ROOT / 'data' / 'raw' / 'diabetic_data.csv'
ids_data_path = ROOT / 'data' / 'raw' / 'IDS_mapping.csv'

df = pd.read_csv(raw_data_path, low_memory=False)
print("diabetic_data.csv loaded successfully!")
print(f"Dataset Shape: {df.shape[0]:,} rows (encounters), {df.shape[1]} columns")


#### Show Top 5 Records (`df.head()`)

In [ ]:
df.head()

#### Show Bottom 5 Records (`df.tail()`)

In [ ]:
df.tail()

### 3. Data Checks to Perform
- Check Missing values & `?` placeholders
- Check Duplicates
- Check Data Types (`df.info()`)
- Check the number of unique values of each column (`df.nunique()`)
- **Check Statistics of dataset (`df.describe()`, `df.describe().T`, and categorical `df.describe(include='O').T`)**
- Explore categories present in categorical features


#### 3.1 Check Missing Values & `?` Placeholders

In [ ]:
# In UCI Diabetes dataset, missing values are denoted by '?'
print("Standard Null (NaN) Values per column:")
print(df.isnull().sum()[df.isnull().sum() > 0])

print("\nMissing placeholders ('?') per column:")
q_marks = (df == '?').sum()
missing_table = pd.DataFrame({
    'Missing_Count': q_marks[q_marks > 0],
    'Percentage': (q_marks[q_marks > 0] / len(df) * 100).round(2)
}).sort_values(by='Missing_Count', ascending=False)
missing_table


#### 3.2 Check Duplicates

In [ ]:
print(f"Exact Duplicate Rows across all 50 columns: {df.duplicated().sum()}")
print(f"Unique Encounters (encounter_id): {df['encounter_id'].nunique():,}")
print(f"Unique Patients (patient_nbr): {df['patient_nbr'].nunique():,}")
print(f"Patients with repeat admissions: {(df['patient_nbr'].value_counts() > 1).sum():,}")


#### 3.3 Check Data Types & Structure (`df.info()`)

In [ ]:
df.info()

#### 3.4 Check Unique Values for Each Column (`df.nunique()`)

In [ ]:
df.nunique()

#### 3.5 Check Statistics of Dataset: Numerical (`df.describe().T`)

In [ ]:
df.describe().T

#### 3.6 Check Statistics of Dataset: Categorical (`df.describe(include='O').T`)

In [ ]:
df.describe(include='O').T

### 4. Exploring Data (Feature Segmentation & Target Formulation)
Separating Numerical and Categorical columns for systematic univariate and bivariate investigation.


In [ ]:
numeric_features = [feature for feature in df.columns if df[feature].dtype != 'O']
categorical_features = [feature for feature in df.columns if df[feature].dtype == 'O']

print(f"We have {len(numeric_features)} Numerical Features: {numeric_features}")
print(f"\nWe have {len(categorical_features)} Categorical Features: {categorical_features}")


#### 4.1 Target Variable Definition & Imbalance Dynamics
The raw target column is `readmitted` with 3 values:
- `<30`: readmitted within 30 days (**Positive Class = 1**)
- `>30`: readmitted after 30 days (**Negative Class = 0**)
- `NO`: not readmitted (**Negative Class = 0**)


In [ ]:
# Original 3-class distribution
print("Original 3-Class Target Distribution:")
print(df['readmitted'].value_counts())
print("\nPercentages:")
print(df['readmitted'].value_counts(normalize=True).round(4) * 100)

# Derive binary target
df['readmitted_30_days'] = (df['readmitted'] == '<30').astype(int)

print("\nDerived Binary Target (readmitted_30_days):")
print(df['readmitted_30_days'].value_counts())
print(f"Positive Class (<30d): {df['readmitted_30_days'].mean()*100:.2f}%")
print(f"Negative Class (NO or >30d): {(1 - df['readmitted_30_days'].mean())*100:.2f}%")
print(f"Majority-class baseline accuracy: {(1 - df['readmitted_30_days'].mean())*100:.2f}%")


### 5. Data Visualization (Univariate & Bivariate Exploration)

#### 5.1 Target Distribution Visualization

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 3-Class
sns.countplot(x='readmitted', data=df, palette=['#3B82F6', '#0D9488', '#EF4444'], ax=axes[0])
axes[0].set_title('Original 3-Class Target Distribution', fontweight='bold')
axes[0].set_xlabel('Readmission Status')
axes[0].set_ylabel('Number of Encounters')

# Binary
sns.countplot(x='readmitted_30_days', data=df, palette=['#1E3A8A', '#EF4444'], ax=axes[1])
axes[1].set_title('Derived Binary Target (30-Day Readmission)', fontweight='bold')
axes[1].set_xticklabels(['No / >30d Readmit (0)', '<30d Readmitted (1)'])
axes[1].set_xlabel('Readmission Target')
axes[1].set_ylabel('Number of Encounters')

plt.tight_layout()
plt.show()


#### 5.2 Demographic Analysis (Age, Gender, Race vs Readmission Rate)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Age
age_rates = df.groupby('age')['readmitted_30_days'].mean() * 100
sns.barplot(x=age_rates.index, y=age_rates.values, color='#0D9488', ax=axes[0])
axes[0].set_title('30-Day Readmission Rate by Age Group', fontweight='bold')
axes[0].set_ylabel('Readmission Rate (%)')
axes[0].tick_params(axis='x', rotation=45)

# Gender
gender_df = df[df['gender'].isin(['Male', 'Female'])]
gender_rates = gender_df.groupby('gender')['readmitted_30_days'].mean() * 100
sns.barplot(x=gender_rates.index, y=gender_rates.values, palette=['#3B82F6', '#EC4899'], ax=axes[1])
axes[1].set_title('30-Day Readmission Rate by Gender', fontweight='bold')
axes[1].set_ylabel('Readmission Rate (%)')

# Race
race_clean = df['race'].replace('?', 'Missing/Unknown')
race_rates = df.groupby(race_clean)['readmitted_30_days'].mean() * 100
sns.barplot(x=race_rates.index, y=race_rates.values, color='#1E3A8A', ax=axes[2])
axes[2].set_title('30-Day Readmission Rate by Race', fontweight='bold')
axes[2].set_ylabel('Readmission Rate (%)')
axes[2].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()


#### 5.3 Prior Healthcare Utilization (Dominant Readmission Predictor)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for idx, (col, title) in enumerate([
    ('number_inpatient', 'Prior Inpatient Admissions'),
    ('number_emergency', 'Prior Emergency Encounters'),
    ('number_outpatient', 'Prior Outpatient Visits')
]):
    capped = df[col].clip(upper=5)
    rates = df.groupby(capped)['readmitted_30_days'].mean() * 100
    sns.barplot(x=rates.index, y=rates.values, color='#1E3A8A', ax=axes[idx])
    axes[idx].set_title(f"{title} (Capped at 5+)", fontweight='bold')
    axes[idx].set_xlabel('Number of Prior Encounters')
    axes[idx].set_ylabel('30-Day Readmission Rate (%)')

plt.tight_layout()
plt.show()


#### 5.4 Clinical Predictors: Length of Stay, Lab Procedures, Medications, Diagnoses

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

sns.boxplot(x='readmitted_30_days', y='time_in_hospital', data=df, palette=['#3B82F6', '#EF4444'], ax=axes[0, 0])
axes[0, 0].set_title('Time in Hospital (Days) vs 30-Day Readmission', fontweight='bold')
axes[0, 0].set_xticklabels(['No / >30d', '<30d Readmitted'])

sns.boxplot(x='readmitted_30_days', y='num_lab_procedures', data=df, palette=['#3B82F6', '#EF4444'], ax=axes[0, 1])
axes[0, 1].set_title('Number of Lab Procedures vs 30-Day Readmission', fontweight='bold')
axes[0, 1].set_xticklabels(['No / >30d', '<30d Readmitted'])

sns.boxplot(x='readmitted_30_days', y='num_medications', data=df, palette=['#3B82F6', '#EF4444'], ax=axes[1, 0])
axes[1, 0].set_title('Number of Medications vs 30-Day Readmission', fontweight='bold')
axes[1, 0].set_xticklabels(['No / >30d', '<30d Readmitted'])

sns.boxplot(x='readmitted_30_days', y='number_diagnoses', data=df, palette=['#3B82F6', '#EF4444'], ax=axes[1, 1])
axes[1, 1].set_title('Number of Diagnoses vs 30-Day Readmission', fontweight='bold')
axes[1, 1].set_xticklabels(['No / >30d', '<30d Readmitted'])

plt.tight_layout()
plt.show()


#### 5.5 Glycemic Control, HbA1c Results & Insulin Titration

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# HbA1c
a1c_rates = df.groupby('A1Cresult')['readmitted_30_days'].mean() * 100
sns.barplot(x=a1c_rates.index, y=a1c_rates.values, color='#0D9488', ax=axes[0])
axes[0].set_title('30-Day Readmission Rate by HbA1c Result', fontweight='bold')
axes[0].set_ylabel('Readmission Rate (%)')

# Insulin
ins_rates = df.groupby('insulin')['readmitted_30_days'].mean() * 100
sns.barplot(x=ins_rates.index, y=ins_rates.values, color='#3B82F6', ax=axes[1])
axes[1].set_title('Readmission Rate by Insulin Status', fontweight='bold')
axes[1].set_ylabel('Readmission Rate (%)')

# Medication Change
chg_rates = df.groupby('change')['readmitted_30_days'].mean() * 100
sns.barplot(x=['No Change', 'Med Changed'], y=chg_rates.values, palette=['#3B82F6', '#F59E0B'], ax=axes[2])
axes[2].set_title('Readmission Rate by Medication Change', fontweight='bold')
axes[2].set_ylabel('Readmission Rate (%)')

plt.tight_layout()
plt.show()


#### 5.6 Correlation Heatmap of Numerical Features

In [ ]:
num_cols = [
    'time_in_hospital', 'num_lab_procedures', 'num_procedures',
    'num_medications', 'number_outpatient', 'number_emergency',
    'number_inpatient', 'number_diagnoses', 'readmitted_30_days'
]
corr_matrix = df[num_cols].corr()

plt.figure(figsize=(10, 8))
sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='coolwarm', vmin=-0.15, vmax=0.55, linewidths=0.5)
plt.title('Correlation Matrix of Numerical Encounter Features', fontweight='bold')
plt.show()


### 6. Conclusions & Next Steps from EDA
1. **Severe Imbalance:** 30-day readmissions represent 11.16% of encounters. Standard accuracy (88.8%) is misleading; models must be optimized using **PR-AUC, ROC-AUC, Sensitivity, and F1**.
2. **Prior Utilization Signal:** Prior inpatient admissions (`number_inpatient`) is the single strongest predictor of 30-day readmissions ($p < 10^{-300}$).
3. **Repeated Patient Visits:** 30,248 encounters are repeat admissions by existing patients. **Patient-Aware Splitting (`GroupShuffleSplit` on `patient_nbr`)** is mandatory to prevent data leakage.
4. **Terminal Discharges:** Deceased/hospice encounters (discharge disposition codes 11, 13, 14, 19, 20, 21) must be excluded from readmission cohorts.
5. **Feature Engineering:** ICD-9 diagnostic codes must be grouped into broad clinical categories (Circulatory, Diabetes, Respiratory, etc.) and medication adjustments synthesized into polypharmacy indices.
